# **MODEL EXPERIMENTATION**
with GCP Integration


In [67]:
#imports 
import sys

sys.path.insert(0, "..")
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats
from scripts.plot_utils import (
    set_theme,
    plot_grid,
    plot_barplot,
    plot_barplot_grid,
    plot_histogram,
    plot_numeric_x_numeric_grid,
    plot_numeric_x_across_categories_grid,
    plot_all_numeric_by_base_category_grid,
    plot_categorical_x_categorical_grid,
)
from scripts.gcs_utils import (
    log_dataset_to_gcs,
    log_pipeline_run,
    register_vertex_dataset,
)
from kfp import compiler
from google.cloud import aiplatform

# Components
from vertex.components import (
    load_validate_data,
    split_data,
    oversample_training,
    fit_apply_preprocessing_v1,
    apply_preprocessing_v1,
    train_model,
    evaluate_model,
)

# Pipelines
from vertex.pipelines import (
    preprocessing_pipeline,
    training_pipeline,
    training_pipeline_no_oversample,
)


In [68]:
# variable declarations
TARGET_COL = "readmission_within_30_days"
ID_COL = "patient_id"
PROJECT_ID = "readmission-543-project"
LOCATION = "us-central1"
BUCKET_ROOT_URI = "gs://readmissions_bucket_v2"


---
## KFP Preprocessing Pipeline

Self-contained KFP v2 components for the preprocessing pipeline.  
Order: `load_validate_data` → `split_data` → `oversample_training` → `fit_apply_preprocessing` → `apply_preprocessing`

In [69]:
# Components are defined in vertex/components/ and imported above:
#   load_validate_data                          ← ingest.py
#   split_data                                  ← split.py
#   oversample_training                         ← oversample.py
#   fit_apply_preprocessing_v1                  ← preprocessing.py
#   apply_preprocessing_v1                      ← preprocessing.py
#   train_model                                 ← train.py
#   evaluate_model                              ← evaluate.py


In [70]:
PREPROCESSING_PIPELINE_JSON = "../vertex/pipelines/readmissions_preprocessing_pipeline.json"

compiler.Compiler().compile(
    pipeline_func=preprocessing_pipeline,
    package_path=PREPROCESSING_PIPELINE_JSON,
)

print(f"Pipeline compiled to {PREPROCESSING_PIPELINE_JSON}")


Pipeline compiled to ../vertex/pipelines/readmissions_preprocessing_pipeline.json


---
## KFP Training Pipeline

`train_model` and `evaluate_model` components extending the preprocessing pipeline.  
Order: `...preprocessing...` → `train_model` → `evaluate_model`

- **`model_type`**: `"logistic"` | `"random_forest"` | `"xgboost"`  
- **`hyperparams_json`**: JSON string of kwargs passed to the chosen estimator (e.g. `'{"n_estimators": 200, "max_depth": 5}'`)

In [71]:
TRAINING_PIPELINE_JSON = "../vertex/pipelines/readmissions_training_pipeline.json"

compiler.Compiler().compile(
    pipeline_func=training_pipeline,
    package_path=TRAINING_PIPELINE_JSON,
)

print(f"Pipeline compiled to {TRAINING_PIPELINE_JSON}")


Pipeline compiled to ../vertex/pipelines/readmissions_training_pipeline.json


---
## Pipeline Submission with Vertex AI Datasets

Register the training CSV as a managed Vertex AI Dataset, then submit the training pipeline.  
The dataset resource name flows through `load_validate_data` metadata → Vertex ML Metadata, giving a native console lineage graph: **Dataset → Pipeline Run → Model**.

- To reuse an existing dataset instead of creating a new one, replace `TabularDataset.create(...)` with `aiplatform.TabularDataset(dataset_name="projects/.../datasets/...")`.

In [72]:
from pathlib import Path

RAW_TRAIN_PATH = Path("../data/raw/healthcare_readmissions_dataset_train.csv")
DATASET_VERSION = "v0.0"
EXPERIMENT_NAME = "readmissions-model-exp"

log_dataset_to_gcs(
    DATASET_LOCAL_PATH=RAW_TRAIN_PATH,
    VERSION_ID=DATASET_VERSION,
    BUCKET_ROOT_URI=BUCKET_ROOT_URI,
    PROJECT_ID=PROJECT_ID,
    LOCATION=LOCATION,
    log_experiment=True,
    EXPERIMENT_NAME=EXPERIMENT_NAME,
    description="Raw training data",
    tags=["raw"],
    resume_run=True,
)


Uploaded: gs://readmissions_bucket_v2/datasets/readmissions/v0.0/train.csv
Uploaded: gs://readmissions_bucket_v2/datasets/readmissions/v0.0/manifest.json


Logged dataset version to Vertex Experiments: readmissions-model-exp / readmissions-data-v0-0


In [73]:
DATASET_VERSION = "v0.0"
dataset, DATASET_GCS_URI = register_vertex_dataset(
    dataset_version=DATASET_VERSION,
    bucket_root_uri=BUCKET_ROOT_URI,
    project_id=PROJECT_ID,
    location=LOCATION,
)
DATASET_RESOURCE_NAME = dataset.resource_name


Reusing existing dataset: projects/182027088454/locations/us-central1/datasets/1513965389040582656
Dataset registered : projects/182027088454/locations/us-central1/datasets/1513965389040582656
GCS source         : gs://readmissions_bucket_v2/datasets/readmissions/v0.0/train.csv


In [74]:
# Submit training pipeline, then log the run (with eval metrics) to Vertex Experiments.
MODEL_VERSION = "v0"
EXPERIMENT_NAME = "readmissions-model-exp"

job = aiplatform.PipelineJob(
    display_name=f"readmissions-training-{DATASET_VERSION}-{MODEL_VERSION}",
    template_path=TRAINING_PIPELINE_JSON,
    pipeline_root=f"{BUCKET_ROOT_URI}/pipelines",
    parameter_values={
        "dataset_gcs_uri": DATASET_GCS_URI,
        "dataset_version": DATASET_VERSION,
        "model_type": "xgboost",
        "hyperparams_json": "{}",
    },
)

job.submit()
print(f"Pipeline submitted: {job.display_name}")

log_pipeline_run(
    pipeline_job=job,
    dataset_version=DATASET_VERSION,
    training_dataset_path=DATASET_GCS_URI,
    model_version=MODEL_VERSION,
    PROJECT_ID=PROJECT_ID,
    LOCATION=LOCATION,
    EXPERIMENT_NAME=EXPERIMENT_NAME,
    wait_for_completion=True,
)


Creating PipelineJob
PipelineJob created. Resource name: projects/182027088454/locations/us-central1/pipelineJobs/readmissions-training-pipeline-20260524120750
To use this PipelineJob in another session:
pipeline_job = aiplatform.PipelineJob.get('projects/182027088454/locations/us-central1/pipelineJobs/readmissions-training-pipeline-20260524120750')
View Pipeline Job:
https://console.cloud.google.com/vertex-ai/locations/us-central1/pipelines/runs/readmissions-training-pipeline-20260524120750?project=182027088454
Pipeline submitted: readmissions-training-v0.0-v0


Associating projects/182027088454/locations/us-central1/metadataStores/default/contexts/readmissions-model-exp-pipeline-run-v0-20260524190751 to Experiment: readmissions-model-exp


Waiting for pipeline to complete...
PipelineJob run completed. Resource name: projects/182027088454/locations/us-central1/pipelineJobs/readmissions-training-pipeline-20260524120750
Logged eval metrics: {'val_precision': 0.6915254237288135, 'val_recall': 0.7311827956989247, 'val_pr_auc': 0.7864636123279664, 'val_roc_auc': 0.9442192501975507, 'val_f1': 0.710801393728223}
Logged pipeline run to Vertex Experiments: readmissions-model-exp / pipeline-run-v0-20260524190751


---
## Dataset v1.0 — Outlier Removal

Remove age and BMI outliers identified during EDA (z-score thresholds: age > 3, BMI > 3.5), save the cleaned CSV locally, and register it as a new versioned dataset in GCS and Vertex AI.


In [75]:
raw_df = pd.read_csv(
    "../data/raw/healthcare_readmissions_dataset_train.csv",
    keep_default_na=False,
    na_values=[""],
)

raw_df.head()

,PatientID,Age,Gender,Ethnicity,Hospital ID,Height (m),Smoker,BMI,Weight (kg),Adjusted Weight (kg),Has Diabetes,Has Hypertension,Exercise Frequency,Diet Type,Number of Prior Visits,Medications Prescribed,Length of Stay,Type of Treatment,Readmission within 30 Days
0,1000000,23,Female,African American,Hosp2,1.6,False,25.0,64.0,63.283346,0,0,Regular,High-fat,3.0,3.0,0,None,0
1,1000002,56,Female,Hispanic,Hosp3,1.8,True,27.0,87.5,87.678859,0,0,Regular,High-fat,2.0,NaN,2,None,0
2,1000003,28,Male,African American,Hosp1,1.8,False,35.0,113.4,113.497844,0,1,None,Other,NaN,2.0,5,None,0
3,1000004,70,Female,Caucasian,Hosp2,1.8,False,27.7,89.7,89.717694,0,0,None,Other,3.0,NaN,0,Major Surgery,0
4,1000005,48,Female,Hispanic,Hosp1,1.9,False,22.4,80.9,80.528927,0,0,Occasional,High-fat,7.0,5.0,7,Major Surgery,1


In [76]:
import scipy.stats as stats
import numpy as np
# age outlier checking
threshold = 3
age_z_scores = np.abs(stats.zscore(raw_df["Age"]))
age_outliers = np.where(age_z_scores > threshold)[0]
print(f"Identified {len(age_outliers)} age outliers at threshold {threshold}:")

display(raw_df.loc[age_outliers, "Age"])


Identified 80 age outliers at threshold 3:


122     158
136     167
253     126
535     147
547     120
       ... 
7251    136
7589    165
7814    195
7976    145
8027    156
Name: Age, Length: 80, dtype: int64

In [77]:
df_transformed = raw_df.drop(index=age_outliers).reset_index(drop=True)

In [78]:
# looking into potential bmi outliers

threshold = 3.5
z_scores = np.abs(stats.zscore(df_transformed['BMI']))
bmi_outliers = np.where(z_scores > threshold)[0]
print(f"Identified {len(bmi_outliers)} BMI outliers at threshold {threshold}:")

display(df_transformed.loc[bmi_outliers, 'BMI'])

Identified 7 BMI outliers at threshold 3.5:


792     43.0
797     43.7
3878    44.0
4444     9.3
4771    43.5
4820    43.8
6398     8.3
Name: BMI, dtype: float64

In [79]:
df_transformed = df_transformed.copy().drop(index=bmi_outliers).reset_index(drop=True)

In [80]:
df_transformed.info()

<class 'pandas.DataFrame'>
RangeIndex: 7951 entries, 0 to 7950
Data columns (total 19 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   PatientID                   7951 non-null   int64  
 1   Age                         7951 non-null   int64  
 2   Gender                      7951 non-null   str    
 3   Ethnicity                   7951 non-null   str    
 4   Hospital ID                 7951 non-null   str    
 5   Height (m)                  7951 non-null   float64
 6   Smoker                      7951 non-null   bool   
 7   BMI                         7951 non-null   float64
 8   Weight (kg)                 7951 non-null   float64
 9   Adjusted Weight (kg)        7951 non-null   float64
 10  Has Diabetes                7951 non-null   int64  
 11  Has Hypertension            7951 non-null   int64  
 12  Exercise Frequency          7951 non-null   str    
 13  Diet Type                   7951 non-null   

In [81]:
df_transformed.to_csv("../data/processed/healthcare_readmissions_dataset_train_no_outliers.csv", index=False)

In [82]:

RAW_TRAIN_PATH = Path("../data/processed/healthcare_readmissions_dataset_train_no_outliers.csv")
DATASET_VERSION = "v1.0"
EXPERIMENT_NAME = "readmissions-model-exp"

log_dataset_to_gcs(
    DATASET_LOCAL_PATH=RAW_TRAIN_PATH,
    VERSION_ID=DATASET_VERSION,
    BUCKET_ROOT_URI=BUCKET_ROOT_URI,
    PROJECT_ID=PROJECT_ID,
    LOCATION=LOCATION,
    log_experiment=True,
    EXPERIMENT_NAME=EXPERIMENT_NAME,
    description="Training data without outliers",
    tags=["no_outliers"],
    resume_run=True,
)


Uploaded: gs://readmissions_bucket_v2/datasets/readmissions/v1.0/train.csv
Uploaded: gs://readmissions_bucket_v2/datasets/readmissions/v1.0/manifest.json


Logged dataset version to Vertex Experiments: readmissions-model-exp / readmissions-data-v1-0


In [83]:
DATASET_VERSION = "v1.0"
dataset, DATASET_GCS_URI = register_vertex_dataset(
    dataset_version=DATASET_VERSION,
    bucket_root_uri=BUCKET_ROOT_URI,
    project_id=PROJECT_ID,
    location=LOCATION,
)
DATASET_RESOURCE_NAME = dataset.resource_name


Reusing existing dataset: projects/182027088454/locations/us-central1/datasets/1453729744024502272
Dataset registered : projects/182027088454/locations/us-central1/datasets/1453729744024502272
GCS source         : gs://readmissions_bucket_v2/datasets/readmissions/v1.0/train.csv


## **XGBoost Experimentation**
- **Baseline**: Default hyperparameters, no oversampling.  
- **Experiment 1**: Default hyperparameters, with oversampling.  
- **Experiment 2**: Hyperparameter tuning with oversampling (e.g. `n_estimators=200`, `max_depth=5`).

### Baseline — XGBoost, No Oversampling

Default XGBoost hyperparameters on the imbalanced training set. Establishes the performance floor before any class-balance intervention.


In [84]:
NO_OVERSAMPLE_TRAINING_PIPELINE_JSON = "../vertex/pipelines/readmissions_training_no_oversample_pipeline.json"

compiler.Compiler().compile(
    pipeline_func=training_pipeline_no_oversample,
    package_path=NO_OVERSAMPLE_TRAINING_PIPELINE_JSON,
)

print(f"Pipeline compiled to {NO_OVERSAMPLE_TRAINING_PIPELINE_JSON}")


Pipeline compiled to ../vertex/pipelines/readmissions_training_no_oversample_pipeline.json


In [85]:
# Submit training pipeline, then log the run (with eval metrics) to Vertex Experiments.
MODEL_VERSION = "v0"
EXPERIMENT_NAME = "xgboost-exp"

job = aiplatform.PipelineJob(
    display_name=f"readmissions-training-{DATASET_VERSION}-{MODEL_VERSION}-baseline-run-1",
    template_path=NO_OVERSAMPLE_TRAINING_PIPELINE_JSON,
    pipeline_root=f"{BUCKET_ROOT_URI}/pipelines",
    parameter_values={
        "dataset_gcs_uri": DATASET_GCS_URI,
        "dataset_version": DATASET_VERSION,
        "model_type": "xgboost",
        "hyperparams_json": "{}",
    },
)

job.submit()
print(f"Pipeline submitted: {job.display_name}")

log_pipeline_run(
    pipeline_job=job,
    dataset_version=DATASET_VERSION,
    training_dataset_path=DATASET_GCS_URI,
    model_version=MODEL_VERSION,
    PROJECT_ID=PROJECT_ID,
    LOCATION=LOCATION,
    EXPERIMENT_NAME=EXPERIMENT_NAME,
    wait_for_completion=True,
)


Creating PipelineJob
PipelineJob created. Resource name: projects/182027088454/locations/us-central1/pipelineJobs/readmissions-training-pipeline-no-oversample-20260524120805
To use this PipelineJob in another session:
pipeline_job = aiplatform.PipelineJob.get('projects/182027088454/locations/us-central1/pipelineJobs/readmissions-training-pipeline-no-oversample-20260524120805')
View Pipeline Job:
https://console.cloud.google.com/vertex-ai/locations/us-central1/pipelines/runs/readmissions-training-pipeline-no-oversample-20260524120805?project=182027088454
Pipeline submitted: readmissions-training-v1.0-v0-baseline-run-1


Associating projects/182027088454/locations/us-central1/metadataStores/default/contexts/xgboost-exp-pipeline-run-v0-20260524190806 to Experiment: xgboost-exp


Waiting for pipeline to complete...
PipelineJob run completed. Resource name: projects/182027088454/locations/us-central1/pipelineJobs/readmissions-training-pipeline-no-oversample-20260524120805
Logged eval metrics: {'val_precision': 0.8, 'val_pr_auc': 0.8430751026773832, 'val_recall': 0.6521739130434783, 'val_roc_auc': 0.9552212486912438, 'val_f1': 0.718562874251497}
Logged pipeline run to Vertex Experiments: xgboost-exp / pipeline-run-v0-20260524190806


### Experiment 1 — XGBoost with SMOTE Oversampling

Same default hyperparameters as baseline, but the training split is oversampled via SMOTE before fitting. Isolates the effect of class balancing on recall.


In [86]:
job = aiplatform.PipelineJob(
    display_name=f"readmissions-training-{DATASET_VERSION}-{MODEL_VERSION}-oversampling-experiment-1-run-1",
    template_path=TRAINING_PIPELINE_JSON,
    pipeline_root=f"{BUCKET_ROOT_URI}/pipelines",
    parameter_values={
        "dataset_gcs_uri": DATASET_GCS_URI,
        "dataset_version": DATASET_VERSION,
        "model_type": "xgboost",
        "hyperparams_json": "{}",
    },
)

job.submit()
print(f"Pipeline submitted: {job.display_name}")

log_pipeline_run(
    pipeline_job=job,
    dataset_version=DATASET_VERSION,
    training_dataset_path=DATASET_GCS_URI,
    model_version=MODEL_VERSION,
    PROJECT_ID=PROJECT_ID,
    LOCATION=LOCATION,
    EXPERIMENT_NAME=EXPERIMENT_NAME,
    wait_for_completion=True,
)

Creating PipelineJob
PipelineJob created. Resource name: projects/182027088454/locations/us-central1/pipelineJobs/readmissions-training-pipeline-20260524120820
To use this PipelineJob in another session:
pipeline_job = aiplatform.PipelineJob.get('projects/182027088454/locations/us-central1/pipelineJobs/readmissions-training-pipeline-20260524120820')
View Pipeline Job:
https://console.cloud.google.com/vertex-ai/locations/us-central1/pipelines/runs/readmissions-training-pipeline-20260524120820?project=182027088454
Pipeline submitted: readmissions-training-v1.0-v0-oversampling-experiment-1-run-1


Associating projects/182027088454/locations/us-central1/metadataStores/default/contexts/xgboost-exp-pipeline-run-v0-20260524190821 to Experiment: xgboost-exp


Waiting for pipeline to complete...
PipelineJob projects/182027088454/locations/us-central1/pipelineJobs/readmissions-training-pipeline-20260524120820 current state:
3
PipelineJob run completed. Resource name: projects/182027088454/locations/us-central1/pipelineJobs/readmissions-training-pipeline-20260524120820
Logged eval metrics: {'val_precision': 0.7473309608540926, 'val_recall': 0.7608695652173914, 'val_pr_auc': 0.8439029493008561, 'val_roc_auc': 0.9575356808287872, 'val_f1': 0.7540394973070018}
Logged pipeline run to Vertex Experiments: xgboost-exp / pipeline-run-v0-20260524190821


### Experiment 2 — XGBoost + SMOTE, Bayesian Hyperparameter Search (Vertex AI Vizier)

Uses Vertex AI Vizier (Gaussian Process Bandit) instead of exhaustive grid search. Vizier proposes hyperparameter combinations, observes `val_roc_auc` after each trial, and uses that signal to guide subsequent suggestions — finding better configurations with fewer total trials than a full grid search.

- **Search space**: `learning_rate`, `n_estimators`, `max_depth`, `subsample`, `colsample_bytree`
- **Algorithm**: `GAUSSIAN_PROCESS_BANDIT` (Bayesian optimization)
- **Parallelism**: `PARALLEL_TRIAL_COUNT` jobs submitted per round; Vizier updates its model between rounds


In [87]:
from google.cloud.aiplatform.vizier import Study, pyvizier as vz
from scripts.gcs_utils import _extract_task_metrics
import json
from datetime import datetime, timezone

HPT_EXPERIMENT_NAME = "xgboost-vizier-exp"
DATASET_VERSION = "v1.0"
DATASET_GCS_URI = f"{BUCKET_ROOT_URI}/datasets/readmissions/{DATASET_VERSION}/train.csv"

# Vizier display names must start with a letter and contain only letters, numbers, and underscores
_version_slug = DATASET_VERSION.replace(".", "_").replace("-", "_")
VIZIER_STUDY_DISPLAY_NAME = f"readmissions_xgboost_hpt_{_version_slug}"

# Configure the search space and objective
problem = vz.StudyConfig()
problem.metric_information.append(
    vz.MetricInformation(name="val_roc_auc", goal=vz.ObjectiveMetricGoal.MAXIMIZE)
)
root = problem.search_space.select_root()
root.add_float_param("learning_rate", min_value=0.01, max_value=0.3, scale_type=vz.ScaleType.LOG)
root.add_int_param("n_estimators", min_value=50, max_value=500)
root.add_int_param("max_depth", min_value=2, max_value=8)
root.add_float_param("subsample", min_value=0.6, max_value=1.0)
root.add_float_param("colsample_bytree", min_value=0.6, max_value=1.0)
# ALGORITHM_UNSPECIFIED lets Vertex AI choose the algorithm — defaults to GP Bandit (Bayesian optimization)
problem.algorithm = vz.Algorithm.ALGORITHM_UNSPECIFIED

# create_or_load resumes an existing study if the display name already exists
study = Study.create_or_load(
    display_name=VIZIER_STUDY_DISPLAY_NAME,
    problem=problem,
    project=PROJECT_ID,
    location=LOCATION,
)
print(f"Study: {study.resource_name}")
print(f"  Display name : {VIZIER_STUDY_DISPLAY_NAME}")
print(f"  Algorithm    : ALGORITHM_UNSPECIFIED (Vertex AI defaults to GP Bandit / Bayesian optimization)")
print(f"  Objective    : maximize val_roc_auc")
print(f"  Params       : learning_rate, n_estimators, max_depth, subsample, colsample_bytree")


Study: projects/182027088454/locations/us-central1/studies/3867079513519
  Display name : readmissions_xgboost_hpt_v1_0
  Algorithm    : ALGORITHM_UNSPECIFIED (Vertex AI defaults to GP Bandit / Bayesian optimization)
  Objective    : maximize val_roc_auc
  Params       : learning_rate, n_estimators, max_depth, subsample, colsample_bytree


In [88]:
MAX_TRIALS = 20
PARALLEL_TRIAL_COUNT = 4   # Vizier suggests N candidates per round; all submitted in parallel

RUN_TIMESTAMP = datetime.now(timezone.utc).strftime("%Y%m%d%H%M%S")
completed_count = 0
failed_count = 0

for round_num in range(MAX_TRIALS // PARALLEL_TRIAL_COUNT):
    print(f"\n--- Round {round_num + 1}/{MAX_TRIALS // PARALLEL_TRIAL_COUNT} ({completed_count} done so far) ---")
    trials = study.suggest(count=PARALLEL_TRIAL_COUNT)

    # Submit all trials in this round before waiting on any
    round_jobs = []
    for i, trial in enumerate(trials):
        hyperparams = {name: param.value for name, param in trial.parameters.items()}
        hyperparams["n_estimators"] = int(hyperparams["n_estimators"])
        hyperparams["max_depth"] = int(hyperparams["max_depth"])

        model_version = f"vizier-r{round_num}-t{i}"
        job = aiplatform.PipelineJob(
            display_name=f"readmissions-vizier-{DATASET_VERSION}-{model_version}-{RUN_TIMESTAMP}",
            template_path=TRAINING_PIPELINE_JSON,
            pipeline_root=f"{BUCKET_ROOT_URI}/pipelines",
            parameter_values={
                "dataset_gcs_uri": DATASET_GCS_URI,
                "dataset_version": DATASET_VERSION,
                "model_type": "xgboost",
                "hyperparams_json": json.dumps(hyperparams),
            },
        )
        job.submit()
        round_jobs.append((job, trial, hyperparams, model_version))
        print(f"  Submitted {model_version}: {hyperparams}")

    # Wait for each job, extract val_roc_auc, and report back to Vizier
    for job, trial, hyperparams, model_version in round_jobs:
        try:
            job.wait()
            metrics = _extract_task_metrics(job, task_name="evaluate-model")
            roc_auc = metrics.get("val_roc_auc")
            if roc_auc is None:
                raise ValueError(f"val_roc_auc not in pipeline output. Got: {metrics}")
            measurement = vz.Measurement()
            measurement.metrics["val_roc_auc"] = vz.Metric(value=roc_auc)
            trial.add_measurement(measurement)
            trial.complete()
            completed_count += 1
            print(f"  ✓ {model_version} | val_roc_auc={roc_auc:.4f}")
        except Exception as e:
            trial.complete(infeasible_reason=str(e)[:500])
            failed_count += 1
            print(f"  ✗ {model_version} FAILED: {e}")

print(f"\nSearch complete — {completed_count} succeeded, {failed_count} failed.")



--- Round 1/5 (0 done so far) ---
Suggest Study study backing LRO: projects/182027088454/locations/us-central1/studies/3867079513519/operations/3867079513519__1
<class 'google.cloud.aiplatform_v1.services.vizier_service.client.VizierServiceClient'>
Study study suggested. Resource name: projects/182027088454/locations/us-central1/studies/3867079513519
Creating PipelineJob
PipelineJob created. Resource name: projects/182027088454/locations/us-central1/pipelineJobs/readmissions-training-pipeline-20260524120842
To use this PipelineJob in another session:
pipeline_job = aiplatform.PipelineJob.get('projects/182027088454/locations/us-central1/pipelineJobs/readmissions-training-pipeline-20260524120842')
View Pipeline Job:
https://console.cloud.google.com/vertex-ai/locations/us-central1/pipelines/runs/readmissions-training-pipeline-20260524120842?project=182027088454
  Submitted vizier-r0-t0: {'colsample_bytree': 0.8, 'learning_rate': 0.05477225575051661, 'max_depth': 5, 'n_estimators': 275, '

### Vizier Results — Ranked Comparison

List all completed trials from the Vizier study, ranked by `val_roc_auc`. The optimal configuration is printed below the table — use it for the final retrain.


In [92]:
import pandas as pd

# Inspect the first trial to understand the actual API before parsing all trials
_all_trials = study.trials()
print(f"Total trials: {len(_all_trials)}")

if _all_trials:
    t = _all_trials[0]
    print(f"\nTrial type: {type(t)}")
    print(f"\nPublic attributes (dir):")
    print([a for a in dir(t) if not a.startswith("__")])
    print(f"\nvars / __dict__:")
    try:
        print(vars(t))
    except TypeError:
        print("(not a dict-based object)")


Total trials: 20

Trial type: <class 'google.cloud.aiplatform.vizier.trial.Trial'>

Public attributes (dir):
['_FutureManager__latest_future', '_FutureManager__latest_future_lock', '_abc_impl', '_are_futures_done', '_assert_gca_resource_is_available', '_complete_future', '_construct_sdk_resource_from_gapic', '_delete', '_delete_method', '_empty_constructor', '_exception', '_format_resource_name', '_format_resource_name_method', '_gca_resource', '_generate_display_name', '_get_and_validate_project_location', '_get_gca_resource', '_getter_method', '_instantiate_client', '_latest_future', '_list', '_list_method', '_list_with_local_order', '_parse_resource_name', '_parse_resource_name_method', '_project_tuple', '_raise_future_exception', '_resource_id_validator', '_resource_is_available', '_resource_noun', '_revisioned_resource_id_validator', '_submit', '_sync_gca_resource', '_sync_object_with_future_result', '_wait_for_resource_creation', 'add_measurement', 'api_client', 'client_class', '